# Vector similarity search and hybrid search

## 1) Imports and functions

In [15]:
import requests
import pdfplumber
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from openai import DefaultHttpxClient, DefaultAsyncHttpxClient
import httpx

load_dotenv()

True

In [2]:
def chunk_text(text, chunk_size, overlap, split_on_whitespace_only=True):
    chunks = []
    index = 0

    while index < len(text):
        if split_on_whitespace_only:
            prev_whitespace = 0
            left_index = index - overlap
            while left_index >= 0:
                if text[left_index] == " ":
                    prev_whitespace = left_index
                    break
                left_index -= 1
            next_whitespace = text.find(" ", index + chunk_size)
            if next_whitespace == -1:
                next_whitespace = len(text)
            chunk = text[prev_whitespace:next_whitespace].strip()
            chunks.append(chunk)
            index = next_whitespace + 1
        else:
            start = max(0, index - overlap + 1)
            end = min(index + chunk_size + overlap, len(text))
            chunk = text[start:end].strip()
            chunks.append(chunk)
            index += chunk_size

    return chunks

In [20]:
# creation of clients
emb_model = OpenAIEmbeddings(
    model=os.getenv("MODEL_EMBEDDING"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
    http_client=DefaultHttpxClient(verify=False),
    http_async_client=DefaultAsyncHttpxClient(verify=False),
)

llm_model = ChatOpenAI(
    model=os.getenv("MODEL_LLM"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("BASE_URL"),
    streaming=True,
    http_client=DefaultHttpxClient(verify=False),
    http_async_client=DefaultAsyncHttpxClient(verify=False),
)

## 2) Load data

In [5]:
remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "./data/ch02-downloaded.pdf"

response = requests.get(remote_pdf_url, verify=False)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
        print("Download concluded!")
else:
    print("Failed to download the PDF. Status code: ", response.status_code)

c:\Users\francisco.bneto\AppData\Local\miniconda3\envs\genai-neo4j-py314\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'arxiv.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\francisco.bneto\AppData\Local\miniconda3\envs\genai-neo4j-py314\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'arxiv.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Download concluded!


In [6]:
text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(f"Extraction successfully completed. Sample: '{text[:50]}...'")

Extraction successfully completed. Sample: 'Einstein’s Patents and Inventions
Asis Kumar Chaud...'


## 3) Chunk creation

In [7]:
chunks = chunk_text(text, 500, 40)
print("Amount of chunks: ", len(chunks))
print("Instance of chunk: ", chunks[0])

Amount of chunks:  89
Instance of chunk:  Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
collaborators had patented a wide variety of inventions


## 4) Embeddings creation

In [13]:
embeddings = emb_model.embed_documents(chunks)
print("Amount of embeddings: ", len(embeddings))
print("dimension of embeddings: ", len(embeddings[0]))

Amount of embeddings:  89
dimension of embeddings:  4096


## 5) Database with vector similarity search function

In [4]:
from neo4j import GraphDatabase

In [7]:
neo4j_driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD"))
)

In [17]:
neo4j_driver.execute_query(
    """
    CREATE VECTOR INDEX pdf IF NOT EXISTS
    FOR (c:Chunk)
    ON c.embedding
    """
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000002343CDC2270>, keys=[])

In [21]:
cypher_query = """
    WITH $chunks AS chunks, range(0, size($chunks)) AS index
    UNWIND index AS i
    WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
    MERGE (c: Chunk {index: i})
    SET c.text = chunk, c.embedding = embedding
    """

neo4j_driver.execute_query(
    cypher_query,
    chunks=chunks,
    embeddings=embeddings
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000002343644DD10>, keys=[])

In [9]:
# check the data into database
records, _, _ = neo4j_driver.execute_query(
    """
    MATCH (c:Chunk) WHERE c.index = 0
    RETURN c.embedding, c.text
    """
)

print(f"TEXT: {records[0]["c.text"][0:200]}...")
print('-----------------------------------------------------')
print(f"EMBEDDING: {records[0]["c.embedding"][0:20]}...")

TEXT: Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Sci...
-----------------------------------------------------
EMBEDDING: [0.023668045178055763, 0.03668547049164772, 0.002248464385047555, 0.009112197905778885, -0.01597592979669571, 0.012780744582414627, -0.08567832410335541, 0.0037425097543746233, -0.008875517174601555, -0.02059119939804077, 0.0045265136286616325, 0.011952362954616547, 0.01751435361802578, 0.03195185959339142, -0.011301491409540176, 0.012603234499692917, 0.03976231813430786, -0.023431364446878433, 0.01988115906715393, -0.012307384051382542]...


## 6) Performing vector search

In [10]:
# Embedding user question
question = "At what time was Einstein really interested in experimental works?"
question_emb = emb_model.embed_query(question)

print(f"QUESTION EMBEDDING: {question_emb[:50]}...")

QUESTION EMBEDDING: [0.015431426465511322, 0.032762106508016586, 0.01436309702694416, -0.007567333988845348, -8.810008876025677e-05, -0.0024927689228206873, -0.037747643887996674, 0.015134667977690697, 0.018636414781212807, -0.002463093027472496, -0.010386536829173565, 0.019348634406924248, 0.022316215559840202, 0.01412568986415863, -0.022791029885411263, -0.013354118913412094, 0.026708237826824188, -0.026233423501253128, 0.032287292182445526, 0.0217227004468441, -0.023978061974048615, 0.024690281599760056, 0.02101048082113266, -0.011336162686347961, -0.024690281599760056, -2.7473317459225655e-05, -0.017211975529789925, -0.01014913059771061, -0.024571578949689865, 0.0017434543697163463, 0.014600503258407116, -0.00036909300251863897, -0.00010803853365359828, 0.01080199796706438, -0.029913226142525673, 0.005282295867800713, 0.019467337056994438, -0.026233423501253128, 0.0041546146385371685, -0.010920701548457146, 0.016855865716934204, 0.005697757471352816, 0.006409977097064257, 0.0040359

In [11]:
# Performing vector search in Neo4j
query = '''
CALL db.index.vector.queryNodes('pdf', 2, $question_embedding)
YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''

similar_records, _, _ = neo4j_driver.execute_query(
    query,
    question_embedding = question_emb
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('pdf', 2, $question_embedding)\nYIELD node AS hits, score\nRETURN hits.text AS text, score, hits.index AS index\n"


In [12]:
for record in similar_records:
    print(f"TEXT: {record['text']}")
    print(f"SCORE: {record['score']} ----- {record['index']}")
    print("\n=====================================================\n")

TEXT: CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, “most of the time I worked in the physical laboratory,
fascinated by the direct contact with observation.” Einstein's
SCORE: 0.8408478498458862 ----- 42


TEXT: in nuances of machines and instruments. However, it must also be
emphasized that his main occupation was theoretical physics. The inventions he worked upon were
his diversions. In his unproductive times he used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was i

## 7) Generating an answer using an LLM

In [18]:
# The LLM context
system_message = """
You're an Einstein expert, but can only use the provided documents to respond to the questions.
"""

user_message = f"""
Use the following documents to answer the question that will follow:
{[doc["text"] for doc in similar_records]}

-------
The question to answer using information only from the above documents:
question: {question}
"""

In [21]:
print("Question: ", question)
messages = [
    ("system", system_message),
    ("user", user_message)
]

print("Response: ", end="")

for chunk in llm_model.stream(messages):
    print(chunk.content, end="", flush=True)

Question:  At what time was Einstein really interested in experimental works?
Response: According to the provided document, Einstein was genuinely interested in experimental works during his days at the ETH (Eidgenössische Technische Hochschule, or Swiss Federal Polytechnic).

## 8) Adding full-text search to the RAG application

### 8.1) Full-text search index

In [22]:
neo4j_driver.execute_query(
    """
    CREATE FULLTEXT INDEX PdfChunkFullText FOR (c:Chunk)
    ON EACH [c.text]
    """
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x0000024D415E1940>, keys=[])

In [23]:
neo4j_driver.execute_query(
    """
    SHOW INDEXES
    """
)

EagerResult(records=[<Record id=3 name='PdfChunkFullText' state='ONLINE' populationPercent=100.0 type='FULLTEXT' entityType='NODE' labelsOrTypes=['Chunk'] properties=['text'] indexProvider='fulltext-2.0' owningConstraint=None lastRead=None readCount=0>, <Record id=1 name='index_1b9dcc97' state='ONLINE' populationPercent=100.0 type='LOOKUP' entityType='RELATIONSHIP' labelsOrTypes=None properties=None indexProvider='token-lookup-1.0' owningConstraint=None lastRead=None readCount=0>, <Record id=0 name='index_460996c0' state='ONLINE' populationPercent=100.0 type='LOOKUP' entityType='NODE' labelsOrTypes=None properties=None indexProvider='token-lookup-1.0' owningConstraint=None lastRead=neo4j.time.DateTime(2026, 11, 17, 16, 58, 22, 298000000, tzinfo=<UTC>) readCount=92>, <Record id=2 name='pdf' state='ONLINE' populationPercent=100.0 type='VECTOR' entityType='NODE' labelsOrTypes=['Chunk'] properties=['embedding'] indexProvider='vector-2026.08' owningConstraint=None lastRead=neo4j.time.DateTi

### 8.2) Performing hybrid search

In [24]:
hybrid_query = """
CALL {
    // vector index
    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding)
    YIELD node, score
    WITH collect({node: node, score: score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // normalize scores
    RETURN n.node AS node, (n.score / max) AS score
    UNION
    // keyboard index
    CALL db.index.fulltext.queryNodes('PdfChunkFullText', $question, {limit: $k})
    YIELD node, score
    WITH collect({node: node, score: score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // normalize scores
    RETURN n.node AS node, (n.score / max) AS score
}
// deduplicate nodes
WITH node, max(score) AS score ORDER BY score DESC LIMIT $k
RETURN node, score
"""

In [39]:
similar_hybrid_records, _, _ = neo4j_driver.execute_query(
    hybrid_query,
    question_embedding=question_emb,
    question=question,
    k=4
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=5, offset=32>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 32, 'line': 4, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL {\n    // vector index\n    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding)\n    YIELD node, score\n    WITH collect({node: node, score: score}) AS nodes, max(score) AS max\n    UNWIND nodes AS n\n    // normalize scores\n    RETURN n.node AS node, (n.score / max) AS score\n    UNION\n    // keyboard index\n    CALL 

In [40]:
for record in similar_hybrid_records:
    print(record["node"]["text"])
    print(f"Score: {record['score']}\tIndex: {record['node']['index']}")
    print("==================================================")

CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, “most of the time I worked in the physical laboratory,
fascinated by the direct contact with observation.” Einstein's
Score: 1.0	Index: 42
in nuances of machines and instruments. However, it must also be
emphasized that his main occupation was theoretical physics. The inventions he worked upon were
his diversions. In his unproductive times he used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three major inven